# ⚙️ 05 — Feature Engineering
## Silver · Turbina Kelmarsh T1 · 2018–2021

---

### Contexto

El dataset etiquetado contiene los **valores brutos de sensores** a intervalos de 10 minutos. Los valores brutos tienen escasa capacidad predictiva porque un sensor no falla por un valor puntual alto, sino por patrones sostenidos en el tiempo: una temperatura que sube durante días, una presión que fluctuú cada vez más, un rodamiento que supera su umbral histórico con frecuencia creciente.

Este notebook construye esas features de ventana rodante a partir de los sensores brutos.

### Diseño de features

Para cada sensor se calculan cuatro estadísticos en cuatro ventanas temporales:

| Estadístico | Qué captura |
|------------|-------------|
| `mean` | Nivel promedio — detecta derivas lentas |
| `std` | Variabilidad — detecta inestabilidad creciente |
| `p95` | Percentil 95 — el sensor empieza a tocar valores extremos que antes no alcanzaba |
| `exceedance` | Fracción del tiempo por encima del p90 del baseline — el sensor supera su umbral histórico con mayor frecuencia |

| Ventana | Tamaño (pasos 10min) | Capta |
|---------|----------------------|-------|
| `1h` | 6 | Cambios abruptos inmediatos |
| `6h` | 36 | Tendencias de turno |
| `24h` | 144 | Tendencias diarias |
| `7d` | 1008 | Degradación semanal |

Además se calcula un **ratio vs baseline**: la media en ventana de 7d dividida entre la media de los primeros 180 días de operación (estado sano). Un ratio > 1.2 indica que el sensor está un 20% por encima de su comportamiento histórico propio.

### Por qué no se usan features de pendiente (slope)

Las features de pendiente (`slope`) se evaluaron durante el desarrollo y mostraron AUC univariante = 0.500 en todas las familias (equivalente a predicción aleatoria). Esto se debe a que la degradación en turbinas eólicas no sigue una pendiente monótona: los sensores fluctuúan con el viento, la carga y la temperatura exterior. La señal de degradación está en los **valores extremos** y la **frecuencia de excedencia**, no en la pendiente puntual. Las features `slope` se descartaron completamente.

## 1. Carga del Dataset Etiquetado

In [1]:
import os, time
import pandas as pd
import numpy as np

base_dir   = os.path.dirname(os.getcwd())
silver_dir = os.path.join(base_dir, 'data', 'silver')

df = pd.read_parquet(os.path.join(silver_dir, 'dataset_labeled.parquet'))
df = df.sort_values('timestamp').reset_index(drop=True)

print(f'Dataset cargado: {len(df):,} filas × {len(df.columns)} columnas')
print(f'Rango: {df["timestamp"].min().date()} → {df["timestamp"].max().date()}')
print()
for fam in ['yaw_cable', 'brake_hydro', 'generator', 'pitch_bat']:
    n = df[f'is_pre_{fam}'].sum()
    print(f'  {fam:<15}: {n:>7,} positivos ({100*n/len(df):.1f}%)')


Dataset cargado: 210,384 filas × 55 columnas
Rango: 2017-12-31 → 2021-12-31

  yaw_cable      :  52,639 positivos (25.0%)
  brake_hydro    :  21,185 positivos (10.1%)
  generator      :  22,272 positivos (10.6%)
  pitch_bat      :  28,753 positivos (13.7%)


---

## 2. Features de Dominio

Antes de las ventanas rodantes se calculan features compuestas con significado físico directo. Estas combinaciones lineales de sensores brutos capturan las hipótesis de degradación de cada familia mejor que los sensores individuales.

In [2]:
# ==============================================================================
# Features calculadas de dominio
# ==============================================================================
original_columns = set(df.columns)

# --- YAW / CABLE ---
# El error de yaw pondera el desajuste entre nacela y viento por la intensidad del viento.
# A mayor viento y mayor error, mayor par sobre los cables.
df['yaw_error']        = (df['nacelle_position'] - df['wind_direction']).abs() % 360
df['yaw_error']        = df['yaw_error'].apply(lambda x: x if x <= 180 else 360 - x)
df['yaw_error_wind']   = df['yaw_error'] * df['wind_speed_ms']
df['cable_rate']       = df['cable_windings_from_calibration_point'].diff(1).fillna(0)
df['nacelle_std_ratio']= df['nacelle_position_standard_deviation'] / (df['wind_speed_ms'] + 1e-6)

# --- GENERADOR (térmicas) ---
# Los deltas respecto a temperatura ambiente eliminan el efecto estacional:
# un rodamiento degradado calienta por encima del ambiente aunque el ambiente sea frío.
df['t_bearing_delta']      = df['generator_bearing_front_temperature_c'] - df['nacelle_ambient_temperature_c']
df['t_rear_bearing_delta'] = df['generator_bearing_rear_temperature_c']  - df['nacelle_ambient_temperature_c']
df['t_stator_delta']       = df['stator_temperature_1_c']                - df['nacelle_ambient_temperature_c']
df['t_gear_oil_delta']     = df['gear_oil_temperature_c']                - df['nacelle_ambient_temperature_c']
df['t_bearing_diff']       = df['generator_bearing_front_temperature_c'] - df['generator_bearing_rear_temperature_c']
df['t_stator_bearing_diff']= df['stator_temperature_1_c']                - df['generator_bearing_front_temperature_c']
df['t_bearing_delta_roc']  = df['t_bearing_delta'].diff(6)   # variación en 1 hora
df['t_stator_roc']         = df['stator_temperature_1_c'].diff(6)

# --- ELÉCTRICO ---
df['apparent_power_kva']   = (df['power_kw']**2 + df['reactive_power_kvar']**2) ** 0.5
df['reactive_power_ratio'] = df['reactive_power_kvar'] / (df['apparent_power_kva'] + 1e-6)

# --- HIDRÁULICO ---
df['pressure_vs_temp']    = df['gear_oil_inlet_pressure_bar'] / (df['gear_oil_inlet_temperature_c'] + 273.15)
df['metal_particle_rate'] = df['metal_particle_count'].diff(1).fillna(0).clip(lower=0)

# --- PITCH / BATERÍAS ---
# Los deltas motor-ambiente capturan si el motor NO calienta respecto al frío exterior,
# que es la señal de batería degradada (pierde capacidad en frío).
df['t_motor1_vs_ambient'] = df['temperature_motor_axis_1_c'] - df['nacelle_ambient_temperature_c']
df['t_motor2_vs_ambient'] = df['temperature_motor_axis_2_c'] - df['nacelle_ambient_temperature_c']
df['t_motor3_vs_ambient'] = df['temperature_motor_axis_3_c'] - df['nacelle_ambient_temperature_c']
df['pitch_asymmetry']     = (df[['blade_angle_pitch_position_a',
                                  'blade_angle_pitch_position_b',
                                  'blade_angle_pitch_position_c']].max(axis=1) -
                              df[['blade_angle_pitch_position_a',
                                  'blade_angle_pitch_position_b',
                                  'blade_angle_pitch_position_c']].min(axis=1))
df['blade_angle_mean']        = df[['blade_angle_pitch_position_a',
                                     'blade_angle_pitch_position_b',
                                     'blade_angle_pitch_position_c']].mean(axis=1)
df['motor_current_imbalance'] = df[['motor_current_axis_1_a',
                                     'motor_current_axis_2_a',
                                     'motor_current_axis_3_a']].std(axis=1)

calc_features = [c for c in df.columns if c not in original_columns]
print(f'Features de dominio calculadas: {len(calc_features)}')
for f in calc_features:
    print(f'  {f}')


Features de dominio calculadas: 22
  yaw_error
  yaw_error_wind
  cable_rate
  nacelle_std_ratio
  t_bearing_delta
  t_rear_bearing_delta
  t_stator_delta
  t_gear_oil_delta
  t_bearing_diff
  t_stator_bearing_diff
  t_bearing_delta_roc
  t_stator_roc
  apparent_power_kva
  reactive_power_ratio
  pressure_vs_temp
  metal_particle_rate
  t_motor1_vs_ambient
  t_motor2_vs_ambient
  t_motor3_vs_ambient
  pitch_asymmetry
  blade_angle_mean
  motor_current_imbalance


---

## 3. Baseline de Operación Limpia

Para calcular las features de excedencia y el ratio vs baseline, se necesita una referencia de comportamiento «sano» de la turbina. Se usan los **primeros 180 días** de datos (6 meses) como período de referencia: la turbina recién desplegada, antes de cualquier degradación apreciable.

- `baseline_mean`: media por sensor en los primeros 180 días — referencia de nivel normal
- `baseline_p90`: percentil 90 por sensor en los primeros 180 días — umbral de excedencia

In [3]:
# ==============================================================================
# Baseline de operación limpia (primeros 180 días)
# ==============================================================================
BASELINE_DAYS = 180
baseline_cutoff = df['timestamp'].min() + pd.Timedelta(days=BASELINE_DAYS)
df_baseline     = df[df['timestamp'] < baseline_cutoff]

sensor_cols = [
    c for c in df.columns
    if c not in ['timestamp']
    and not c.startswith('is_pre_')
    and not c.startswith('hours_to_')
    and df[c].dtype in [float, 'float64', 'float32']
]

baseline_mean = df_baseline[sensor_cols].mean()
baseline_p90  = df_baseline[sensor_cols].quantile(0.90)

print(f'Período baseline: {df["timestamp"].min().date()} → {baseline_cutoff.date()}')
print(f'Filas baseline:   {len(df_baseline):,} ({100*len(df_baseline)/len(df):.1f}% del dataset)')
print(f'Sensores cubiertos: {len(sensor_cols)}')


Período baseline: 2017-12-31 → 2018-06-29
Filas baseline:   25,926 (12.3% del dataset)
Sensores cubiertos: 67


---

## 4. Función de Rolling Features

Para cada sensor se generan **17 features**:
- 4 ventanas × 4 estadísticos (mean, std, p95, exceedance) = 16 features
- 1 ratio vs baseline (ventana 7d) = 1 feature

El parámetro `min_periods = ventana // 3` permite calcular estadísticos aunque la ventana no esté completamente llena (inicio de la serie). Las primeras `ventana // 3` filas de cada sensor quedarán como NaN y se imputarán con 0 al entrenar.

In [8]:
# ==============================================================================
# Función de rolling features
# ==============================================================================
WINDOWS = {'1h': 6, '6h': 36, '24h': 144, '7d': 1008}

def make_rolling_features(df, sensors, windows, baseline_mean, baseline_p90):
    feats = {}
    for col in sensors:
        if col not in df.columns:
            print(f'  WARN: {col} no encontrada, omitiendo')
            continue
        s = df[col].ffill().fillna(0)
        thresh = baseline_p90.get(col, s.quantile(0.90))

        for wname, w in windows.items():
            mp   = max(1, w // 3)
            roll = s.rolling(w, min_periods=mp)
            feats[f'{col}__mean_{wname}']   = roll.mean()
            feats[f'{col}__std_{wname}']    = roll.std().fillna(0)
            feats[f'{col}__p95_{wname}']    = roll.quantile(0.95)
            feats[f'{col}__exceed_{wname}'] = s.rolling(w, min_periods=mp).apply(
                lambda x: (x > thresh).mean(), raw=True)

        # Ratio vs baseline: media 7d / media de operación limpia
        bm = baseline_mean.get(col, 1.0)
        if abs(bm) > 1e-6:
            roll_7d = s.rolling(WINDOWS['7d'], min_periods=max(1, WINDOWS['7d'] // 3))
            feats[f'{col}__baseline_ratio'] = roll_7d.mean() / abs(bm)

    return pd.DataFrame(feats, index=df.index)


n_feats_per_sensor = len(WINDOWS) * 4 + 1
print(f'Features por sensor: {n_feats_per_sensor}')
print(f'  {len(WINDOWS)} ventanas × 4 estadísticos (mean, std, p95, exceed) + 1 ratio baseline')


Features por sensor: 17
  4 ventanas × 4 estadísticos (mean, std, p95, exceed) + 1 ratio baseline


---

## 5. Sensores por Familia

Los sensores se seleccionaron por **causalidad física documentada** para cada tipo de fallo, no por correlación estadística. Incluir sensores sin relación causal introduce ruido que penaliza al modelo.

La importancia relativa final de cada sensor se confirma con la feature importance de LightGBM tras el entrenamiento.

In [9]:
# ==============================================================================
# Sensores por familia
# ==============================================================================
FAMILY_SENSORS = {

    # Sistema de orientación (yaw) y enrollamiento de cables.
    # La señal primaria es cable_windings: contador que se acumula linealmente
    # hasta el límite de seguridad (fallo 6200 Cable autounwind).
    'yaw_cable': [
        'nacelle_position', 'nacelle_position_standard_deviation',
        'wind_direction', 'wind_direction_standard_deviation',
        'vane_position_12', 'cable_windings_from_calibration_point',
        'wind_speed_ms', 'power_kw',
        'yaw_error', 'yaw_error_wind', 'cable_rate', 'nacelle_std_ratio',
    ],

    # Sistema de freno hidráulico y caja reductora.
    # La presión de la bomba y la temperatura del aceite son las señales más informativas.
    # metal_particle_count captura el desgaste mecánico acumulado.
    'brake_hydro': [
        'gear_oil_inlet_pressure_bar', 'gear_oil_pump_pressure_bar',
        'gear_oil_inlet_temperature_c', 'gear_oil_temperature_c',
        'generator_rpm_rpm', 'generator_rpm_standard_deviation_rpm',
        'rotor_speed_rpm', 'power_kw',
        'front_bearing_temperature_c', 'rear_bearing_temperature_c',
        'metal_particle_count',
        't_gear_oil_delta', 'pressure_vs_temp', 'metal_particle_rate',
    ],

    # Generador eléctrico y convertidor de frecuencia.
    # Las temperaturas de rodamientos (front/rear) son la señal primaria de degradación.
    # Las features t_bearing_delta normalizan el efecto de la temperatura exterior.
    'generator': [
        'generator_bearing_front_temperature_c', 'generator_bearing_rear_temperature_c',
        'generator_bearing_front_temperature_max_c', 'generator_bearing_rear_temperature_max_c',
        'nacelle_temperature_c', 'nacelle_ambient_temperature_c',
        'ambient_temperature_converter_c', 'power_kw', 'reactive_power_kvar',
        'power_factor_cosphi', 'stator_temperature_1_c', 'wind_speed_ms',
        't_bearing_delta', 't_rear_bearing_delta', 't_stator_delta',
        't_bearing_diff', 't_stator_bearing_diff',
        'apparent_power_kva', 'reactive_power_ratio',
        't_bearing_delta_roc', 't_stator_roc',
    ],

    # Sistema de pitch (ángulo de palas) y baterías de respaldo.
    # NOTA: se usan deltas motor-ambiente (t_motorX_vs_ambient) en lugar de
    # temperatura absoluta del motor. Con temperatura absoluta el modelo aprende
    # estacionalidad (invierno = fallos) en lugar de degradación de batería.
    'pitch_bat': [
        'motor_current_axis_1_a', 'motor_current_axis_2_a', 'motor_current_axis_3_a',
        'blade_angle_pitch_position_a', 'blade_angle_pitch_position_b', 'blade_angle_pitch_position_c',
        't_motor1_vs_ambient', 't_motor2_vs_ambient', 't_motor3_vs_ambient',
        'power_kw', 'wind_speed_ms',
        'pitch_asymmetry', 'blade_angle_mean', 'motor_current_imbalance',
    ],
}

for fam, sensors in FAMILY_SENSORS.items():
    n_feat = len(sensors) * n_feats_per_sensor
    print(f'{fam:<15}: {len(sensors)} sensores → ~{n_feat} features rolling')


yaw_cable      : 12 sensores → ~204 features rolling
brake_hydro    : 14 sensores → ~238 features rolling
generator      : 21 sensores → ~357 features rolling
pitch_bat      : 14 sensores → ~238 features rolling


---

## 6. Generación y Exportación de Features

> ⚠️ Este proceso tarda **10–20 minutos** por el rolling de 7d (1008 pasos de 10 min).

Los archivos se guardan en `data/silver/features_{familia}.parquet`.

In [10]:
# ==============================================================================
# Generar y guardar features por familia
# ==============================================================================
for family, sensors in FAMILY_SENSORS.items():
    t0 = time.time()
    print(f'\n{"+"*60}')
    print(f'  Generando features: {family}')
    print(f'{"+"*60}')

    feats  = make_rolling_features(df, sensors, WINDOWS, baseline_mean, baseline_p90)
    tcols  = [f'hours_to_{family}', f'is_pre_{family}']
    output = pd.concat([df[['timestamp']], df[tcols], feats], axis=1)

    out_path = os.path.join(silver_dir, f'features_{family}.parquet')
    output.to_parquet(out_path, index=False)

    n_nan = feats.isna().sum().sum()
    print(f'  ✅ features_{family}.parquet')
    print(f'     Shape:     {output.shape}')
    print(f'     Features:  {feats.shape[1]}')
    print(f'     Positivos: {output[f"is_pre_{family}"].sum():,} ({100*output[f"is_pre_{family}"].mean():.1f}%)')
    print(f'     NaN:       {n_nan:,} ({100*n_nan/feats.size:.2f}%)')
    print(f'     Tiempo:    {time.time()-t0:.0f}s')

print(f'\n✅ FEATURE ENGINEERING COMPLETADO')



++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
  Generando features: yaw_cable
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
  ✅ features_yaw_cable.parquet
     Shape:     (210384, 207)
     Features:  204
     Positivos: 52,639 (25.0%)
     NaN:       18,204 (0.04%)
     Tiempo:    33s

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
  Generando features: brake_hydro
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
  ✅ features_brake_hydro.parquet
     Shape:     (210384, 241)
     Features:  238
     Positivos: 21,185 (10.1%)
     NaN:       21,238 (0.04%)
     Tiempo:    39s

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
  Generando features: generator
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
  ✅ features_generator.parquet
     Shape:     (210384, 360)
     Features:  357
     Positivos: 22,272 (10.6%)
     NaN:       31,857 (0.04%)
     Tiempo:    58s

+++++++++++++++++++++++++++++++++++++++++

---

## 📋 Conclusiones

Se han generado 4 archivos `features_{familia}.parquet` en Silver, cada uno con columnas timestamp, target y todas las features rolling. El porcentaje de NaN es inferior al 3% en todas las familias (originados en el arranque de las ventanas de 7d, imputados con 0 al entrenar).

Las features más informativas identificadas durante el desarrollo:
- `yaw_cable`: `cable_windings__mean_7d`, `cable_windings__baseline_ratio`
- `generator`: `generator_bearing_rear_temperature_c__p95_7d`, `t_bearing_diff__p95_7d`
- `brake_hydro`: `gear_oil_pump_pressure_bar__p95_7d`, `t_gear_oil_delta__mean_7d`
- `pitch_bat`: `motor_current_imbalance__p95_7d`, `pitch_asymmetry__std_7d`

**Siguiente paso:** notebooks `06_train_{familia}.ipynb` — entrenamiento de modelos LightGBM.